# Gold business analysis
Descriptive customer snapshot analysis using published Gold tables only. No causal, profitability or time-series claims are supported.

In [ ]:
from pathlib import Path
import os,sys,json
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
PROJECT=Path.cwd()
if PROJECT.name=='notebooks': PROJECT=PROJECT.parent
os.chdir(PROJECT)
import pandas as pd
from IPython.display import display,Markdown
manifest=json.loads((PROJECT/'data/reports/latest_gold.json').read_text())
folder=Path(manifest['gold_path'])
parquet=Path(manifest['parquet_path'])
tables={p.stem:pd.read_parquet(p) for p in parquet.glob('*.parquet')}
print('Gold build:',manifest['gold_build_id'])
display(tables['mart_customer_kpis'].T)

## Country scale, balances and coverage
Compare observed denominators before interpreting differences.

In [ ]:
country=tables['mart_country_summary']
display(country[['country','total_customers','total_balance','average_balance','average_customer_value_score','balance_coverage_pct','score_coverage_pct','high_value_customer_pct']].sort_values('total_balance',ascending=False))

## Age, credit and tenure patterns
Unknown values remain visible. Small cohorts may have unstable averages.

In [ ]:
for name in ['mart_age_group_summary','mart_credit_band_summary','mart_tenure_summary']:
    print(name)
    display(tables[name].sort_values('sort_order'))

In [ ]:
fact=tables['fact_customer_financials']
pairs=fact[['tenure','balance']].dropna()
print({'tenure_balance_complete_pairs':len(pairs),'pearson_r':pairs.corr().iloc[0,1]})
print('Descriptive association; no evidence of a causal tenure effect.')

## Balance concentration and outliers
Percentile cohorts use observed balances only. Extreme values are flagged rather than removed.

In [ ]:
display(tables['mart_balance_concentration'])
display(tables['mart_outlier_summary'])

## Value tiers, primary segments and profiles
Value is a relative analytical proxy. Segment priority makes primary groups exclusive; flags overlap.

In [ ]:
customer=tables['dim_customer']
display(customer.customer_value_segment.value_counts().rename('customers').to_frame())
display(customer.primary_customer_segment.value_counts().rename('customers').to_frame())
display(tables['mart_customer_profiles'])
display(tables['mart_model_parameters'].T)

In [ ]:
segments=tables['mart_customer_segments']
display(segments.groupby(['country','primary_customer_segment']).agg(customers=('customer_key','size'),high_value_customers=('is_high_value','sum'),average_value=('customer_value_score','mean')).reset_index())
display(tables['mart_value_evaluation'])

## Supported findings and checks
Distinguish facts and observations from hypotheses and recommendations. Scenario category changes use the fixed baseline tier thresholds.

In [ ]:
display(Markdown((folder/'business_insights.md').read_text(encoding='utf-8')))
display(pd.read_parquet(Path(manifest['quality_path'])/'gold_validation_report.parquet'))
display(pd.read_csv(Path(manifest['quality_path'])/'sql_python_parity.csv'))